# 文本分类实例

## Step1 导入相关包

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

## Step2 加载数据集

In [4]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/7766 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

## Step3 划分数据集

In [5]:
datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 777
    })
})

## Step4 数据集预处理

In [7]:
import torch

tokenizer = AutoTokenizer.from_pretrained("../hfl/rbt3")

def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/6988 [00:00<?, ? examples/s]

Map:   0%|          | 0/777 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 777
    })
})

## Step5 创建模型

In [8]:
model = AutoModelForSequenceClassification.from_pretrained("../hfl/rbt3")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
model.config

BertConfig {
  "_name_or_path": "../hfl/rbt3",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 3,
  "output_past": true,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "transformers_version": "4.42.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 21128
}

## Step6 创建评估函数

In [2]:
import evaluate

acc_metric = evaluate.load("../../evaluate/metrics/accuracy")
f1_metric = evaluate.load("../../evaluate/metrics/f1")

In [10]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

## Step7 创建TrainingArguments

In [22]:
train_args1 = TrainingArguments(output_dir="./checkpoints1")
train_args1

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=no,
evaluation_strategy=None,
fp16=False,
fp16_backend=auto,
fp16

In [12]:
train_args = TrainingArguments(output_dir="./checkpoints",      # 输出文件夹
                               per_device_train_batch_size=64,  # 训练时的batch_size
                               per_device_eval_batch_size=128,  # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               eval_strategy="epoch",     # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
evaluation_strategy=None,
fp16=False,
fp16_backend=auto,
fp

## Step8 创建Trainer

In [14]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [20]:
trainer

## Step9 模型训练

In [15]:
trainer.train()

  0%|          | 0/330 [00:00<?, ?it/s]

{'loss': 0.635, 'grad_norm': 2.75545072555542, 'learning_rate': 1.9393939393939395e-05, 'epoch': 0.09}
{'loss': 0.586, 'grad_norm': 1.962188720703125, 'learning_rate': 1.8787878787878792e-05, 'epoch': 0.18}
{'loss': 0.535, 'grad_norm': 1.8444409370422363, 'learning_rate': 1.8181818181818182e-05, 'epoch': 0.27}
{'loss': 0.4717, 'grad_norm': 3.0935378074645996, 'learning_rate': 1.7575757575757576e-05, 'epoch': 0.36}
{'loss': 0.3988, 'grad_norm': 2.976560354232788, 'learning_rate': 1.6969696969696972e-05, 'epoch': 0.45}
{'loss': 0.3457, 'grad_norm': 2.7952661514282227, 'learning_rate': 1.6363636363636366e-05, 'epoch': 0.55}
{'loss': 0.3398, 'grad_norm': 4.868321895599365, 'learning_rate': 1.575757575757576e-05, 'epoch': 0.64}
{'loss': 0.3201, 'grad_norm': 6.393346309661865, 'learning_rate': 1.5151515151515153e-05, 'epoch': 0.73}
{'loss': 0.3397, 'grad_norm': 3.2756330966949463, 'learning_rate': 1.4545454545454546e-05, 'epoch': 0.82}
{'loss': 0.3184, 'grad_norm': 3.5924336910247803, 'learn

  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.3006172776222229, 'eval_accuracy': 0.8687258687258688, 'eval_f1': 0.907608695652174, 'eval_runtime': 1.195, 'eval_samples_per_second': 650.222, 'eval_steps_per_second': 5.858, 'epoch': 1.0}
{'loss': 0.272, 'grad_norm': 2.769374132156372, 'learning_rate': 1.2727272727272728e-05, 'epoch': 1.09}
{'loss': 0.2877, 'grad_norm': 2.9101738929748535, 'learning_rate': 1.2121212121212122e-05, 'epoch': 1.18}
{'loss': 0.2756, 'grad_norm': 3.8317437171936035, 'learning_rate': 1.1515151515151517e-05, 'epoch': 1.27}
{'loss': 0.2975, 'grad_norm': 2.483443021774292, 'learning_rate': 1.0909090909090909e-05, 'epoch': 1.36}
{'loss': 0.252, 'grad_norm': 2.45918607711792, 'learning_rate': 1.0303030303030304e-05, 'epoch': 1.45}
{'loss': 0.2766, 'grad_norm': 2.537025213241577, 'learning_rate': 9.696969696969698e-06, 'epoch': 1.55}
{'loss': 0.2675, 'grad_norm': 3.685264825820923, 'learning_rate': 9.090909090909091e-06, 'epoch': 1.64}
{'loss': 0.2611, 'grad_norm': 4.328124523162842, 'learning_rat

  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.272459477186203, 'eval_accuracy': 0.8777348777348777, 'eval_f1': 0.9126034958601656, 'eval_runtime': 1.1502, 'eval_samples_per_second': 675.545, 'eval_steps_per_second': 6.086, 'epoch': 2.0}
{'loss': 0.2598, 'grad_norm': 2.5436573028564453, 'learning_rate': 6.060606060606061e-06, 'epoch': 2.09}
{'loss': 0.2413, 'grad_norm': 2.786320209503174, 'learning_rate': 5.4545454545454545e-06, 'epoch': 2.18}
{'loss': 0.2204, 'grad_norm': 4.38554573059082, 'learning_rate': 4.848484848484849e-06, 'epoch': 2.27}
{'loss': 0.2235, 'grad_norm': 2.7818729877471924, 'learning_rate': 4.242424242424243e-06, 'epoch': 2.36}
{'loss': 0.2312, 'grad_norm': 4.2348952293396, 'learning_rate': 3.6363636363636366e-06, 'epoch': 2.45}
{'loss': 0.2225, 'grad_norm': 2.7904298305511475, 'learning_rate': 3.0303030303030305e-06, 'epoch': 2.55}
{'loss': 0.2324, 'grad_norm': 2.740997552871704, 'learning_rate': 2.4242424242424244e-06, 'epoch': 2.64}
{'loss': 0.2082, 'grad_norm': 4.756917476654053, 'learning_ra

  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.2807876467704773, 'eval_accuracy': 0.8828828828828829, 'eval_f1': 0.9179440937781785, 'eval_runtime': 1.1823, 'eval_samples_per_second': 657.215, 'eval_steps_per_second': 5.921, 'epoch': 3.0}
{'train_runtime': 101.3813, 'train_samples_per_second': 206.784, 'train_steps_per_second': 3.255, 'train_loss': 0.3064710552042181, 'epoch': 3.0}


TrainOutput(global_step=330, training_loss=0.3064710552042181, metrics={'train_runtime': 101.3813, 'train_samples_per_second': 206.784, 'train_steps_per_second': 3.255, 'total_flos': 351909933963264.0, 'train_loss': 0.3064710552042181, 'epoch': 3.0})

## Step10 模型评估

In [16]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.2807876467704773,
 'eval_accuracy': 0.8828828828828829,
 'eval_f1': 0.9179440937781785,
 'eval_runtime': 1.1311,
 'eval_samples_per_second': 686.963,
 'eval_steps_per_second': 6.189,
 'epoch': 3.0}

## Step11 模型预测

In [17]:
trainer.predict(tokenized_datasets["test"])

  0%|          | 0/7 [00:00<?, ?it/s]

PredictionOutput(predictions=array([[ 1.1837693, -1.0005112],
       [-1.8717026,  2.5001292],
       [ 2.2036617, -2.0287411],
       ...,
       [-2.1340554,  2.5201707],
       [-2.062672 ,  2.5531673],
       [ 1.4541856, -1.1907238]], dtype=float32), label_ids=array([0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0,
       0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1,
    

In [18]:
from transformers import pipeline

id2_label = id2_label = {0: "差评！", 1: "好评！"}
model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [19]:
sen = "我觉得不错！"
pipe(sen)

[{'label': '好评！', 'score': 0.9849564433097839}]